# HealthConnect Rwanda: Week 8 Final Model and Data Science Package

> **Track:** Data Science  
> **Client:** HealthConnect Rwanda  
> **Author:** Dorian NJAMI  
> **Programme:** AnalystLab Africa Experience Lab  
> **Project phase:** Week 8 - Final Integration, Documentation and Presentation

## Project Overview

This notebook continues the Week 6 and Week 7 work. It summarizes the Week 7 model tests, incorporates the Data Analytics validation summary, audits the admission-target lineage in this repository, and records the final integration decision.

The Data Analytics summary reports no admission records in its 27 source tables and says the target is blocked. The Week 6 repository inventory differs: it lists 30 logical tables from 37 files and workbook sheets, including an `Admissions` sheet. That sheet has 5,000 rows, but only 586 `AppointmentID` values match the 5,000 appointment records; 4,414 do not. The Week 6 master generated `admission_indicator` from those matches. Until the source versions and keys are reconciled, Week 7 scores are provisional results on an unconfirmed label.

## Notebook Structure

### Part I - Week 7 to Week 8 Transition
1. Summarize Week 7 results, refinements and remaining issues.
2. Include Data Analytics findings that affect model decisions.

### Part II - Week 8 Final Integration and Model Readiness
1. Audit the target against source records.
2. Set prediction-time and feature requirements.
3. Record final model selection, cross-track dependencies and ML Engineering requirements.
4. Prepare the non-technical summary and individual presentation outline.

This notebook documents an analytical prototype and readiness decision. It does not authorize clinical use or claim no-show prediction capability.

## Part I - Week 7 to Week 8 Transition

The Week 8 brief asks for a short transition, not a repeat of the Week 7 report.

| Transition item | Week 7 to Week 8 summary |
|---|---|
| Main Week 7 output | Tested admission classifier workflow comparing a prior-probability reference, Week 6-style Logistic Regression, refined Logistic Regression and Random Forest. |
| Main test result | On the Nov-Dec holdout, refined Logistic Regression AP was 0.124 and ROC-AUC 0.466; prior reference AP was 0.134 and ROC-AUC 0.500. |
| Major issue | The model did not beat the simple reference; the target's source lineage was not confirmed. |
| Refinements | Removed the direct target copy and constant feature, retained appointment rows, added strictly prior appointment history, used chronological splits and compared two model types. |
| Component ready | The model test and evidence package is ready for review. No predictive model is approved for operational use. |
| Still to integrate | Reconcile Analytics source inventory with the repository Admissions sheet; confirm target meaning, keys, and feature timestamps; then retest if justified. |
| Tracks involved | Data Analytics and Data Science for source/target review; ML Engineering for any later pipeline; Project Management for readiness and presentation. |
| Week 8 demonstration | Show the model comparison, source-lineage audit, current no-go decision, and evidence needed to reopen model selection. |

**Interpretation.** Week 7 did not demonstrate model improvement. Week 8 also identifies a target-lineage discrepancy. Do not present the earlier scores as validated admission-prediction performance until the target and join keys are confirmed.

### Week 7 Results Retained for Traceability

These scores came from the Week 7 notebook and use the label then available in the cleaned master. They are retained as historical results, not treated as validated performance on confirmed ground truth.

| Candidate | Test average precision | Test ROC-AUC | Test recall at 0.50 | Week 7 interpretation |
|---|---:|---:|---:|---|
| Prior-probability reference | 0.134 | 0.500 | 0.000 | No ranking ability; AP equals prevalence. |
| Week 6-style Logistic Regression | 0.131 | 0.467 | 0.473 | Did not improve on the reference. |
| Refined Logistic Regression | 0.124 | 0.466 | 0.445 | Chosen on validation AP but below the reference on final test. |
| Refined Random Forest | 0.138 | 0.493 | 0.055 | AP near prevalence; severe train-validation overfitting. |

The paired bootstrap interval for refined Logistic Regression's AP difference versus the prior reference was -0.023 to 0.017 and included zero. No meaningful improvement was demonstrated. The target audit in this notebook adds a stronger qualification: the label itself needs reconciliation.

### Data Analytics Findings Relevant to Data Science

The validation document reports the following findings. They are partner-reported here and have not all been recalculated in this notebook.

| Finding | Data Science implication |
|---|---|
| KPI reconciliation and date-slicer checks passed. | Supports the reported dashboard measures, not the model target. |
| Matched-bill amounts were similar to unmatched bills (KS p=0.66); status and severity shares differed by under one percentage point. | Billing results can be generalized cautiously with the matched-subset caveat; billing is not added to this admission model. |
| Matched monthly billing trend slope was -0.3M/month (p=0.37), with 5.9% variation. | No sustained billing trend was demonstrated; it is not an attendance trend. |
| Facility rating tests p=0.79/0.80 and department rating p=0.29 were not significant. | Exclude rating measures. `departmentname` is a category, not a rating. |
| Revenue Summary 1.13bn did not reconcile to 9.31bn billing and was removed from the dashboard. | Do not use that summary as a model feature or reconciled KPI. |
| No attendance/no-show field; no admission records in the report's 27 tables. | No no-show model is possible; reconcile target source and inventory. |
| Gender/chronic condition considered usable; age, prior appointment count, visit type, department, month, weekday usable with caveats; diagnosis/severity held for timing review. | Retain only provisionally and confirm availability at prediction time. Previous-admission history is on hold until source validation. |

Feedback linkage was 1,756 of 21,064 responses (about 8%). Linked/unlinked complaint rates were 40.9%/40.4%; mean ratings 2.93/2.99 (p=0.05). This may support a broader project insight with the linkage caveat, but feedback is not a predictor here.

## Part II - Week 8 Final Integration and Model Readiness

### 2.1 - Admission Target and Source-Lineage Audit

A complete binary field is not automatically a valid outcome. Compare the source records and appointment keys used to create the label, then reconcile that evidence with the Data Analytics source inventory.

In [1]:
from pathlib import Path

import pandas as pd

# Locate the project root whether Jupyter starts here or in the Notebook folder.
possible_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((root for root in possible_roots
                     if (root / "Data/Cleaned_Datasets/healthconnect_master_clean.csv").exists()), None)
if project_root is None:
    raise FileNotFoundError("Could not locate the project's cleaned master dataset.")

master_path = project_root / "Data/Cleaned_Datasets/healthconnect_master_clean.csv"
source_path = project_root / "Data/Original_Datasets/HC_Clinical_Transactions.xlsx"
master = pd.read_csv(master_path, low_memory=False)
appointments = pd.read_excel(source_path, sheet_name="Appointments")
admissions = pd.read_excel(source_path, sheet_name="Admissions")

# Compare appointment keys using trimmed string values.
appointment_keys = set(appointments["AppointmentID"].dropna().astype("string").str.strip())
admission_keys = admissions["AppointmentID"].dropna().astype("string").str.strip()
matched = admission_keys.isin(appointment_keys)
linked_keys = set(admission_keys.loc[matched])

# Recreate the Week 6 rule: a matched admission row makes an appointment positive.
source_label = master["appointmentid"].astype("string").str.strip().isin(linked_keys)
master_label = pd.to_numeric(master["admission_indicator"], errors="coerce")

lineage_audit = pd.DataFrame([
    {"Check": "Appointment rows", "Observed": len(appointments)},
    {"Check": "Admissions worksheet rows", "Observed": len(admissions)},
    {"Check": "Unique AdmissionID values", "Observed": admissions["AdmissionID"].nunique()},
    {"Check": "Admissions rows linked to an appointment", "Observed": int(matched.sum())},
    {"Check": "Admissions rows with unmatched AppointmentID", "Observed": int((~matched).sum())},
    {"Check": "Distinct linked appointment IDs", "Observed": len(linked_keys)},
    {"Check": "Positive labels in cleaned master", "Observed": int(master_label.sum())},
    {"Check": "Master label equals recreated link flag for every row",
     "Observed": bool(master_label.eq(source_label.astype("int8")).all())},
])
print("Source-lineage audit:\n" + lineage_audit.to_string(index=False))

Source-lineage audit:
                                                Check Observed
                                     Appointment rows     5000
                            Admissions worksheet rows     5000
                            Unique AdmissionID values     5000
             Admissions rows linked to an appointment      586
         Admissions rows with unmatched AppointmentID     4414
                      Distinct linked appointment IDs      586
                    Positive labels in cleaned master      586
Master label equals recreated link flag for every row     True


**Interpretation of the lineage audit.** In the repository's source workbook, the `Admissions` worksheet has 5,000 rows. Only 586 `AppointmentID` values match the 5,000 appointment rows; 4,414 do not match. The cleaned master has 586 positive labels and reproduces the linked-key rule. This confirms how the Week 6 label was produced, but it does not establish that the source rows or links mean a real admission outcome.

The Data Analytics summary describes a different inventory: 27 tables and no admissions, while our Week 6 inventory lists 30 logical tables from 37 files/sheets. The difference may reflect source versions, counting conventions, or a missing workbook. Until Data Analytics and the data owner reconcile the exact files, key definitions, and outcome meaning, the target remains **unverified** and Week 7 scores remain provisional. Do not interpret unmatched rows as non-admissions without confirmation.

### 2.2 - Prediction Time and Feature Decisions

The proposed prediction point is **after the initial clinical assessment and before the admission decision/outcome**, once every selected field has actually been recorded. This timing is a design proposal that requires confirmation from the clinical/data owner. A booking-time model cannot use a diagnosis or severity value recorded later; an arrival-time model requires a separate timestamped feature set.

| Feature | Week 8 decision | Reason |
|---|---|---|
| Gender, chronic condition | Potentially usable | Data Analytics reports these as usable; verify definitions and missingness. |
| Age, prior appointment count, visit type, department, month and weekday | Potentially usable with caveats | Confirm each field is available before the proposed prediction point and define history windows. |
| Diagnosis and severity | Hold | Same-visit timing is unclear; use only if timestamps show they precede the admission decision. |
| Prior admission count/rate | Exclude pending source validation | These depend on the disputed Admissions rows and cannot be trusted yet. |
| Facility/department ratings | Exclude | Data Analytics found no significant differences; department name itself is a separate category. |
| Billing, payments, revenue, complaint/feedback | Exclude from this model | Different outcome and potentially observed after prediction; revenue summary failed reconciliation. |
| `admissions_record_count` | Prohibit as a predictor | It duplicates the existing label and leaks it directly. |
| Attendance/no-show outcome | Unavailable | The validation summary reports no such field; new validated outcome data are required. |

The Week 7 feature set is not approved unchanged. Resolve the target first, then rebuild and temporally retest using only confirmed pre-outcome variables.

### 2.3 - Final Candidate Selection

**No predictive model is approved as the final candidate.** The Week 7 refined Logistic Regression scored below the prior reference on final-test average precision (0.124 vs 0.134) and ROC-AUC (0.466 vs 0.500); its paired bootstrap interval included zero. The Random Forest showed a large train-validation gap and did not establish a stable gain.

The label is now also under source review: the repository join yields 586 matched positives and 4,414 unmatched admission keys, while Data Analytics reports no admission records in its 27-table inventory. Until this is resolved, final model selection and ML integration are blocked. The final Data Science outcome is a documented **no-go/readiness decision**, a traceable comparison, and a precise data requirement for reopening model development.

### 2.4 - Final Integration Readiness and HC-POD Record

| Readiness item | Current response | Status |
|---|---|---|
| Final component | Week 7 model-testing evidence plus Week 8 source-lineage and readiness review | Evidence package ready to present |
| Problem addressed | Whether current data can support reliable admission prediction relevant to the broader appointment-support goal | No-show objective remains unsupported |
| Current model status | No final candidate approved; Week 7 metrics use a provisional label | Blocked for predictive use |
| Week 7 improvement | Leakage checks, chronological evaluation, candidate comparison and error analysis | Completed; gain not demonstrated |
| Dependency on Data Analytics/data owner | Reconcile its 27-table inventory with our Admissions worksheet; confirm source version, key mapping, target meaning and field timestamps | Pending |
| Output for ML Engineering | Once resolved: versioned feature schema, prediction time, fitted pipeline, input/output contract and validation summary | Not ready until target is confirmed |
| Readiness evidence | Week 7 outputs, source audit, Data Analytics validation summary and issue register | Analytical review ready; model integration not ready |
| Remaining limitations | 4,414 unmatched admission keys, source-inventory conflict, no no-show label, diagnosis/severity timing, one-year history | Open |

#### Cross-Track Integration Record

| Required item | Week 8 record |
|---|---|
| Track collaborated with | Data Analytics (Agu Precious); Data Science; ML Engineering dependency identified; Project Management for presentation coordination |
| Output received | Data Analytics validation summary with dashboard, billing, rating and target findings |
| Output to provide | Week 7 scores and this repository audit: 5,000 Admissions rows, 586 matched keys and 4,414 unmatched keys. The audit still needs to be sent to the partner/data owner. |
| Final integration activity | Reconcile files, versions, keys and target meaning; confirm prediction time and supported features together |
| What changed | Week 7 candidate downgraded to provisional; prior-admission features held; no model approved as final |
| Improvement to the solution | Prevents an unverified label or broken join from being carried into a final model or ML pipeline |
| Evidence | Partner summary and executed audit in Section 2.1; save the exchange and owner decision with project evidence |
| Retest | Pending target/key resolution; retest with the confirmed outcome and same temporal design |

The Data Analytics exchange has surfaced an actionable issue, but a joint reconciliation and retest have not yet happened. Record those activities as pending until evidence is received.

### 2.5 - ML Engineering Requirements

The current handoff is a readiness notice, not a deployable model contract.

| Requirement | Current specification/status |
|---|---|
| Target/output | Admission probability is proposed only; target semantics and source lineage need confirmation. |
| Prediction time | Proposed after initial assessment and before admission decision; workflow timestamps need confirmation. |
| Final model | None approved. Week 7 Logistic Regression is a provisional historical result. |
| Feature schema | Provisional. Hold prior-admission variables, diagnosis and severity until source/timing checks pass. |
| Attendance/no-show | Unsupported because no validated label was found. |
| Integration gate | Wait for a confirmed target, feature contract, temporal retest and end-to-end test before using predictions in a workflow. |
| Reproducibility | Preserve source version, feature definitions, split dates, dependency versions, fitted preprocessing and evaluation outputs when a valid model exists. |

After target validation, Data Science should provide the serialized pipeline, exact input columns/types, missing and unseen-category behavior, output definition, threshold rationale, version details and reproducible validation evidence.

### 2.6 - Non-Technical Summary and Business Relevance

**What we can say:** Week 7 compared several models against a simple reference using an admission label created from the master data. The refined Logistic Regression did not outperform that reference on the later test period. Data Analytics then reported that its source inventory contains no admission records, while our repository contains an Admissions worksheet with many appointment keys that do not match. We must reconcile the source before trusting those scores as real admission-prediction performance.

**What the project cannot claim:** This work does not predict missed appointments. No attendance/no-show outcome is available. It also does not provide a clinically validated admission recommendation or a model ready for operational use.

**Business value now:** The review found a source and linkage issue before an unverified outcome could be integrated into a final system. It also connects Analytics findings to model design: ratings and unreconciled revenue summaries stay out of the model, and feature timing must be agreed across tracks. The next decision is to confirm the source files, keys and outcome with Data Analytics and the data owner.

### 2.7 - Individual Video and HC-POD Walkthrough Plan

The brief requires an individual 5-10 minute video and an HC-POD integration walkthrough. This outline targets about seven minutes and uses evidenced work only.

| Time | Topic | Evidence to show |
|---|---|---|
| 0:00-0:45 | Introduction and problem | Name, Data Science track; distinguish attendance goal from available admission label. |
| 0:45-2:15 | Week 6-7 contribution | Appointment integration, label construction, leakage review, temporal comparisons. |
| 2:15-3:30 | Results and refinement | Baseline, Logistic Regression, Random Forest; no Week 7 gain established. |
| 3:30-5:00 | Week 8 integration issue | Data Analytics 27-table inventory vs repository audit: 5,000 Admissions rows, 586 matches, 4,414 unmatched. Explain why approval is blocked. |
| 5:00-6:15 | Collaboration and decision | Actual Analytics findings; pending source reconciliation and ML Engineering dependency. |
| 6:15-7:00 | Business value and close | No no-show model, no final model approved, and next evidence required. |

For the HC-POD walkthrough, show how the Analytics validation, Data Science audit, future ML Engineering pipeline and Project Management readiness tracking connect. If a partner test or final integration has not happened, label it pending and present the dependency rather than claiming completion.

## Week 8 Data Science Project Summary

| Required summary item | Current record |
|---|---|
| Final Data Science component | Week 7 model-test evidence and Week 8 target-lineage/readiness decision |
| Week 7 result | Refined Logistic Regression AP 0.124/AUC 0.466; prior reference AP 0.134/AUC 0.500, both on provisional label |
| Most important new finding | Repository Admissions worksheet has 5,000 rows; only 586 keys match and 4,414 do not. Data Analytics reports no admissions in its 27-table inventory. |
| Final candidate | None approved pending target reconciliation |
| Cross-track work | Analytics validation summary received; source audit prepared; reconciliation and retest pending |
| Business relevance | Prevents an unverified outcome from being presented as a usable model; no-show prediction is unsupported |
| Remaining limitations | Conflicting source inventories, unmatched keys, target semantics, diagnosis/severity timing, one-year history, no attendance label |
| Required next action | Reconcile the source package/version, confirm keys and target with the data owner, agree prediction time, rebuild and retest after validation |
| Readiness decision | Evidence package ready for presentation; predictive model and ML Engineering integration are not ready |

This package summarizes Week 7 without repeating its full report. It records a defensible Week 8 outcome and keeps the remaining cross-track dependency explicit.